# 00 · SQL Server → MinIO (`landing-zone`, CSV)

Extrai **todas as tabelas** do banco `seguradora` (SQL Server) e grava cada uma como CSV no bucket `landing-zone` do MinIO.

```mermaid
flowchart LR
    A[(SQL Server<br/>seguradora)] -->|JDBC| B[Spark]
    B -->|CSV| C[(MinIO<br/>landing-zone)]
```

Pré-requisito: `docker compose up -d` (SQL Server + MinIO no ar).

In [1]:
from spark_delta_minio_sqlserver import pipeline
from spark_delta_minio_sqlserver.spark_session import create_session

spark = create_session("00-extracao-landing")
spark.sparkContext.setLogLevel("ERROR")

26/10/11 12:44:08 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 12:44:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/isafel/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c35237a-48cb-4905-9e26-b7a01be9ecad;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central


	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
:: resolution report :: resolve 222ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   7   |   0   |   0   |   0   ||   7   |   0   |
	--------------------------------------------

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Buckets `landing-zone` e `bronze`

In [2]:
pipeline.criar_buckets()

## Tabelas encontradas no SQL Server

In [3]:
tabelas = pipeline.listar_tabelas(spark)
tabelas

['apolice', 'carro', 'cliente', 'marca']

## Extração: uma pasta CSV por tabela

In [4]:
contagem = pipeline.extrair_para_landing(spark)
contagem

{'apolice': 8, 'carro': 8, 'cliente': 4, 'marca': 6}

Conferência: lendo de volta o CSV gravado no MinIO.

In [5]:
spark.read.option("header", True).csv("s3a://landing-zone/carro").show()

+-------+---------+-----------+--------+----+-------+----------+
|  placa|   modelo|     chassi|marca_id| ano|    cor|cliente_id|
+-------+---------+-----------+--------+----+-------+----------+
|ALD3834|     CLIO|34574215969|       1|2011| BRANCO|         1|
|CCR8096|    CRETA|88547875547|       2|2020| BRANCO|         1|
|DLA3438|    PUNTO|98823483434|       3|2013|  PRETO|         2|
|EEE1056|ECO SPORT|56753453455|       4|2020|   AZUL|         2|
|FFR1234|    PALIO|32383478747|       3|2020|AMARELO|         3|
|GQY6753|      S10|72004160549|       5|2015|  PRETO|         3|
|IAC8974|   TIGUAN|77130757746|       6|2020|   AZUL|         4|
|JIE0952|   PASSAT|87493270405|       6|2016|  CINZA|         4|
+-------+---------+-----------+--------+----+-------+----------+



Os arquivos também podem ser vistos no console do MinIO: http://127.0.0.1:9001 (`minioadmin` / `minioadmin`).